In [1]:
import sys, os, logging, gc
import numpy as np
from scipy import optimize

from astropy.cosmology import Planck18
import py21cmfast as p21c
from py21cmfast import cache_tools

is_josh = True
if is_josh:

    os.environ['DM21CM_DIR'] = '/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/DM21cm/'
    os.environ['DM21CM_DATA_DIR'] =  '/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/Data01/'

    os.environ['DH_DIR'] = '/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/DarkHistory/'
    os.environ['DH_DATA_DIR'] = '/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/DarkHistory/DHData/'

    os.environ['P21C_CACHE_DIR'] = './TestCache/'
    
import numpy as np
from scipy import interpolate
from astropy.cosmology import Planck18
import astropy.units as u

from jax import config
config.update("jax_enable_x64", True)
import jax.numpy as jnp

import py21cmfast as p21c
from py21cmfast import cache_tools

sys.path.append(os.environ['DH_DIR'])
from darkhistory.spec.spectrum import Spectrum

sys.path.append(os.environ['DM21CM_DIR'])
import dm21cm.physics as phys
from dm21cm.dh_wrappers import DarkHistoryWrapper, TransferFunctionWrapper
from dm21cm.utils import load_h5_dict
from dm21cm.data_cacher import Cacher
from dm21cm.profiler import Profiler

logging.getLogger().setLevel(logging.INFO)
logging.getLogger('21cmFAST').setLevel(logging.CRITICAL+1)
logging.getLogger('py21cmfast._utils').setLevel(logging.CRITICAL+1)
logging.getLogger('py21cmfast.wrapper').setLevel(logging.CRITICAL+1)

#######################################
###   Import and Construct DM21cm   ###
#######################################

cache_dir = os.environ['P21C_CACHE_DIR']

os.environ['P21C_CACHE_DIR'] = cache_dir
p21c.config['direc'] = os.environ['P21C_CACHE_DIR']
WDIR = os.environ['DM21CM_DIR']
sys.path.append(WDIR)
from dm21cm.dm_params import DMParams
#from dm21cm.evolve import evolve

use_tqdm = True

/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/21cmFAST/src/py21cmfast/_cfg.py:58: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn(
/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/21cmFAST/src/py21cmfast/_cfg.py:42: UserWarning: Your configuration file is out of date. Updating...
  warnings.warn("Your configuration file is out of date. Updating...")


In [2]:
from dm21cm.evolve import get_z_edges, split_xray, gen_injection_boxes, get_r_shells

In [3]:
run_name = 'test'
z_start = 45.
z_end = 5.
dm_params = DMParams(
    mode='decay',
    primary='phot_delta',
    m_DM=1e8, # [eV]
    lifetime=1e31, # [s]
)
enable_elec = False

p21c_initial_conditions = p21c.initial_conditions(
    user_params = p21c.UserParams(
        HII_DIM = 32,
        BOX_LEN = 192, # [conformal Mpc]
        N_THREADS = 1,
    ),
    cosmo_params = p21c.CosmoParams(
        OMm = Planck18.Om0,
        OMb = Planck18.Ob0,
        POWER_INDEX = Planck18.meta['n'],
        SIGMA_8 = Planck18.meta['sigma8'],
        hlittle = Planck18.h,
    ),
    random_seed = 54321,
    write = True,
)
# p21c_astro_params = p21c.AstroParams._defaults_
# astro_params = p21c_astro_params
astro_params = None
p21c_astro_params = None
clear_cache = True
use_DH_init = False
no_injection = False
tf_on_device = True
rerun_DH = False
use_xray_interp_shell = True

/global/scratch/projects/pc_heptheory/fosterjw/21CM_Project/21cmFAST/src/py21cmfast/inputs.py:487: UserWarning: The USE_INTERPOLATION_TABLES setting has changed in v3.1.2 to be default True. You can likely ignore this warning, but if you relied onhaving USE_INTERPOLATION_TABLES=False by *default*, please set it explicitly. To silence this warning, set it explicitly to True. Thiswarning will be removed in v4.
  warnings.warn(


In [4]:
def p21c_step(perturbed_field, spin_temp, ionized_box,
             input_heating=None, input_ionization=None, input_jalpha=None, astro_params=astro_params):

    spin_temp = p21c.spin_temperature(
        perturbed_field = perturbed_field,
        previous_spin_temp = spin_temp,
        input_heating_box = input_heating,
        input_ionization_box = input_ionization,
        input_jalpha_box = input_jalpha,
        astro_params = astro_params,
    )

    ionized_box = p21c.ionize_box(
        perturbed_field = perturbed_field,
        previous_ionize_box = ionized_box,
        spin_temp = spin_temp,
        astro_params = astro_params,
    )

    brightness_temp = p21c.brightness_temperature(
        ionized_box = ionized_box,
        perturbed_field = perturbed_field,
        spin_temp = spin_temp,
    )

    return spin_temp, ionized_box, brightness_temp

In [5]:
data_dir = os.environ['DM21CM_DATA_DIR']
cache_dir = os.environ['P21C_CACHE_DIR'] + '/' + run_name
p21c.config['direc'] = cache_dir
logging.info(f"Cache dir: {cache_dir}")
os.makedirs(cache_dir, exist_ok=True)
if clear_cache:
    cache_tools.clear_cache()
gc.collect()

#===== initialize =====
#--- physics parameters ---
abscs = load_h5_dict(f"{data_dir}/abscissas.h5")
dm_params.set_inj_specs(abscs)

EPSILON = 1e-6
p21c.global_params.Z_HEAT_MAX = z_start + EPSILON
p21c.global_params.ZPRIME_STEP_FACTOR = abscs['zplusone_step_factor']

box_dim = p21c_initial_conditions.user_params.HII_DIM
box_len = p21c_initial_conditions.user_params.BOX_LEN
cosmo = Planck18

#--- DarkHistory and transfer functions ---
tf_wrapper = TransferFunctionWrapper(
    box_dim = box_dim,
    abscs = abscs,
    prefix = data_dir,
    enable_elec = enable_elec,
    on_device = tf_on_device,
)

INFO:root:Cache dir: ./TestCache//test
INFO:jax._src.xla_bridge:Unable to initialize backend 'cuda': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:jax._src.xla_bridge:Unable to initialize backend 'rocm': module 'jaxlib.xla_extension' has no attribute 'GpuAllocatorConfig'
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
INFO:root:TransferFunctionWrapper: Loaded photon transfer functions.
INFO:root:TransferFunctionWrapper: Skipping electron transfer functions.


In [6]:
#--- xray ---
xray_cacher = Cacher(box_dim=box_dim, dx=box_len/box_dim)
xray_cacher.clear_cache()

#--- redshift stepping ---
z_edges = get_z_edges(z_start, z_end, p21c.global_params.ZPRIME_STEP_FACTOR)

#===== initial steps =====
dh_wrapper = DarkHistoryWrapper(dm_params, prefix=p21c.config[f'direc'])

# We have to synchronize at the second step because 21cmFAST acts weird in the first step:
# - global_params.TK_at_Z_HEAT_MAX is not set correctly (it is probably set and evolved for a step)
# - global_params.XION_at_Z_HEAT_MAX is not set correctly (it is probably set and evolved for a step)
# - first step ignores any values added to spin_temp.Tk_box and spin_temp.x_e_box
z_match = z_edges[1]
if use_DH_init:
    dh_wrapper.evolve(end_rs=(1+z_match)*0.9, rerun=rerun_DH)
    T_k_DH_init, x_e_DH_init, phot_bath_spec = dh_wrapper.get_init_cond(rs=1+z_match)
else:
    phot_bath_spec = Spectrum(abscs['photE'], np.zeros_like(abscs['photE']), spec_type='N', rs=1+z_match) # [ph / Bavg]

In [7]:
perturbed_field = p21c.perturb_field(redshift=z_edges[1], init_boxes=p21c_initial_conditions, write = True)
spin_temp, ionized_box, brightness_temp = p21c_step(perturbed_field=perturbed_field, spin_temp=None, ionized_box=None, astro_params=p21c_astro_params)
if use_DH_init:
    spin_temp.Tk_box += T_k_DH_init - np.mean(spin_temp.Tk_box)
    spin_temp.x_e_box += x_e_DH_init - np.mean(spin_temp.x_e_box)
    ionized_box.xH_box = 1 - spin_temp.x_e_box

#===== main loop =====
z_edges = z_edges[1:] # Maybe fix this later
z_range = range(len(z_edges)-1)
records = []
if use_tqdm:
    from tqdm import tqdm
    z_range = tqdm(z_range)
profiler = Profiler()

#--- trackers ---
i_xray_loop_start = 0 # where we start looking for annuli

  0%|          | 0/204 [00:00<?, ?it/s]

In [ ]:
#--- loop ---
for i_z in z_range:

    print_str = f'i_z={i_z}/{len(z_edges)-2} z={z_edges[i_z]:.2f}'

    #===== physical quantities =====
    z_current = z_edges[i_z]
    z_next = z_edges[i_z+1]
    dt = phys.dt_step(z_current, np.exp(abscs['dlnz']))

    #--- for interpolation ---
    delta_plus_one_box = 1 + np.asarray(perturbed_field.density)
    x_e_box = np.asarray(1 - ionized_box.xH_box)
    T_k_box = np.asarray(spin_temp.Tk_box)
    tf_wrapper.init_step(rs=1+z_current, delta_plus_one_box=delta_plus_one_box, x_e_box=x_e_box, T_k_box=T_k_box)

    #--- for dark matter ---
    nBavg = phys.n_B * (1+z_current)**3 # [Bavg / (physical cm)^3]
    rho_DM_box = delta_plus_one_box * phys.rho_DM * (1+z_current)**3 # [eV/(physical cm)^3]
    inj_per_Bavg_box = phys.inj_rate(rho_DM_box, dm_params) * dt * dm_params.struct_boost(1+z_current) / nBavg # [inj/Bavg]

    #===== photon injection and energy deposition =====

    profiler.start()

    if not no_injection:

        #--- xray interpolating shell ---
        if use_xray_interp_shell:

            if len(xray_cacher.states)==0:
                # This is the first step. In the second step, we need to interpolate prior to the
                # first step's state and something. By doing the trapz integration, it is consistent
                # for us to put an all zero state in this step, since there is no emission.
                zero_spectrum = Spectrum(abscs['photE'], np.zeros_like(abscs['photE']), spec_type='N', rs=1+z_current)
                xray_cacher.cache(z_current-1, z_current, zero_spectrum, np.zeros((box_dim, box_dim, box_dim)))
                
            else:
                
                ##########################
                ###   Begin New Code   ###
                ##########################
                
                # conformal distance [cMpc] of z from current shell
                r_from_z = np.vectorize(lambda z: phys.conformal_dx_between_z(z_current, z)) 

                # inverse of r_z
                z_from_r = interpolate.interp1d(r_from_z(np.geomspace(z_current, 200., 1000)),
                                                np.geomspace(z_current, 200., 1000),
                                                bounds_error=False, fill_value='extrapolate') 
                
                # Get the list of r-shells we smooth over
                r_shells = get_r_shells(box_dim, box_len, n_target=40) # R_a in paper
                cached_r = r_from_z(xray_cacher.z_s)
    
                # Don't smooth farther back then we have data
                if np.amax(cached_r) < np.amax(r_shells): 
                    r_shells = np.unique(np.minimum(np.amax(cached_r), r_shells))
                    
                # Go past box_len/2 to the next redshift cache in the interest of ending on a state
                else:
                    r_shells = np.union1d(r_shells, np.amin(cached_r[np.where(cached_r > np.amax(r_shells))]))
    
                # (Min, Max) pairs of the radii on which we smooth
                r_pairs = np.stack((r_shells[:-1], r_shells[1:]), axis = -1)
            
                # The midpoint smoothing radius and associated redshift
                r_mid = (r_shells[1:] + r_shells[:-1]) / 2 # The midpoint of the smoothing
                z_mid = z_from_r(r_mid) # Z for the midpoint of the smoothing edge
                
                # The delta-z of the smoothing interval. Used to go from dEdz -> \Delta E
                dz = np.diff(z_from_r(r_pairs), axis = 1) 

                # Do the interpolated smoothing loop
                for i in range(len(z_mid)):
                    
                    # Deciding on the interpolation nodes
                    locs = np.where(xray_cacher.z_s <= z_mid[i])[0]
                    left_z = np.amax(xray_cacher.z_s[locs])  
                    
                    locs = np.where(xray_cacher.z_s > z_mid[i])[0]
                    right_z =  np.amin(xray_cacher.z_s[locs])
                    
                    # Data at interpolation nodes
                    ftdEdz_right, rel_spec_right = xray_cacher.get_ftdEdz_spec(right_z)
                    ftdEdz_left,  rel_spec_left  = xray_cacher.get_ftdEdz_spec(left_z)
                    
                    # Linear interpolation weights
                    left_weight = (right_z - z_mid[i]) / (right_z - left_z)
                    right_weight = 1-left_weight
                    
                    # Weighted emissivity box
                    ftdEdz = left_weight * ftdEdz_left + right_weight * ftdEdz_right
                    rel_spec = left_weight * rel_spec_left + right_weight * rel_spec_right
                    
                    # Do the smoothing and injection
                    dE = xray_cacher.smooth_box(ftdEdz, r_pairs[i, 0], r_pairs[i, 1])*dz[i]
                    tf_wrapper.inject_phot(rel_spec, inject_type='xray', weight_box=dE)
                    
                # We engineered our smoothing radii to end on a cached state. Now we dump everything prior
                # to that cached state because it will never get used                
                print('Dumping states before:', z_from_r(np.amax(r_pairs)))
                print('Radius of dumped states will be larger than:', np.amax(r_pairs))
                phot_bath_spec += xray_cacher.release_to_bath_prior_to(z_from_r(np.amax(r_pairs)))
                
                ########################
                ###   End New Code   ###
                ########################

        profiler.record('xray')

        #--- bath and homogeneous portion of xray ---
        tf_wrapper.inject_phot(phot_bath_spec, inject_type='bath')

        #--- dark matter (on-the-spot) ---
        tf_wrapper.inject_from_dm(dm_params, inj_per_Bavg_box)

        profiler.record('bath+dm')

    #===== 21cmFAST step =====
    perturbed_field = p21c.perturb_field(redshift=z_next, init_boxes=p21c_initial_conditions)
    input_heating, input_ionization, input_jalpha = gen_injection_boxes(z_next, p21c_initial_conditions)
    tf_wrapper.populate_injection_boxes(input_heating, input_ionization, input_jalpha, dt,)
    spin_temp, ionized_box, brightness_temp = p21c_step(
        perturbed_field, spin_temp, ionized_box,
        input_heating = input_heating,
        input_ionization = input_ionization,
        input_jalpha = input_jalpha,
        astro_params = p21c_astro_params
    )

    profiler.record('21cmFAST')

    #===== prepare spectra for next step =====
    #--- bath (separating out xray) ---
    prop_phot_N = np.array(tf_wrapper.prop_phot_N) # propagating and emitted photons have been stored in tf_wrapper up to this point, time to get them out
    emit_phot_N = np.array(tf_wrapper.emit_phot_N)
    emit_bath_N, emit_xray_N = split_xray(emit_phot_N, abscs['photE'])
    phot_bath_spec = Spectrum(abscs['photE'], prop_phot_N + emit_bath_N, rs=1+z_current, spec_type='N') # photons not emitted to the xray band are added to the bath (treated as uniform)
    phot_bath_spec.redshift(1+z_next)

    #--- xray ---
    x_e_for_attenuation = 1 - np.mean(ionized_box.xH_box)
    attenuation_arr = np.array(tf_wrapper.attenuation_arr(rs=1+z_current, x=np.mean(x_e_for_attenuation))) # convert from jax array
    xray_cacher.advance_spectra(attenuation_arr, z_next)

    xray_spec = Spectrum(abscs['photE'], emit_xray_N, rs=1+z_current, spec_type='N') # [ph/Bavg]
    xray_spec.redshift(1+z_next)
    xray_tot_eng = np.dot(abscs['photE'], emit_xray_N)
    if xray_tot_eng == 0.:
        xray_rel_eng_box = np.zeros_like(tf_wrapper.xray_eng_box)
    else:
        xray_rel_eng_box = tf_wrapper.xray_eng_box / xray_tot_eng # [1 (relative energy)/Bavg]
    if not no_injection:
        if use_xray_interp_shell:
            xray_cacher.cache(z_current, z_next, xray_spec, xray_rel_eng_box)
        else:
            xray_cacher.cache(z_next, xray_rel_eng_box, xray_spec)

    #===== calculate and save some quantities =====
    dE_inj_per_Bavg = dm_params.eng_per_inj * np.mean(inj_per_Bavg_box) # [eV/Bavg]
    dE_inj_per_Bavg_unclustered = dE_inj_per_Bavg / dm_params.struct_boost(1+z_current) # [eV/Bavg]

    records.append({
        'z'   : z_next,
        'T_s' : np.mean(spin_temp.Ts_box), # [mK]
        'T_b' : np.mean(brightness_temp.brightness_temp), # [K]
        'T_k' : np.mean(spin_temp.Tk_box), # [K]
        'x_e' : np.mean(spin_temp.x_e_box), # [1]
        '1-x_H' : np.mean(1 - ionized_box.xH_box), # [1]
        'E_phot' : phot_bath_spec.toteng(), # [eV/Bavg]
        'phot_N' : phot_bath_spec.N, # [ph/Bavg]
        'dE_inj_per_B' : dE_inj_per_Bavg, # [eV/Bavg]
        'dE_inj_per_Bavg_unclustered' : dE_inj_per_Bavg_unclustered, # [eV/Bavg]
        'dep_ion'  : np.mean(tf_wrapper.dep_box[...,0] + tf_wrapper.dep_box[...,1]), # [eV/Bavg]
        'dep_exc'  : np.mean(tf_wrapper.dep_box[...,2]), # [eV/Bavg]
        'dep_heat' : np.mean(tf_wrapper.dep_box[...,3]), # [eV/Bavg]
        'x_e_slice' : np.array(spin_temp.x_e_box[0]), # [1]
        'x_H_slice' : np.array(ionized_box.xH_box[0]), # [1]
        'T_k_slice' : np.array(spin_temp.Tk_box[0]), # [K]
    })

    profiler.record('prep_next')

    if not use_tqdm:
        print(print_str, flush=True)

  0%|          | 1/204 [00:10<34:43, 10.27s/it]

Dumping states before: 44.67847966153375
Radius of dumped states will be larger than: 11.705454725075208


  1%|          | 2/204 [00:26<45:47, 13.60s/it]

Dumping states before: 44.6784830334757
Radius of dumped states will be larger than: 23.46974508923346


  1%|▏         | 3/204 [00:45<54:19, 16.21s/it]

...

  3%|▎         | 6/204 [01:53<1:10:01, 21.22s/it]

Dumping states before: 44.678473363740956
Radius of dumped states will be larger than: 71.12110635441073


  3%|▎         | 7/204 [02:18<1:13:49, 22.48s/it]

Dumping states before: 44.678483445178955
Radius of dumped states will be larger than: 83.18397029747645


  4%|▍         | 8/204 [02:44<1:16:49, 23.52s/it]